# EEG-Flow — Concatenate epochs across runs

Concatenates run-1 and run-2 epoch files per participant, equalizes bad
channels across runs, interpolates, and saves the merged epochs and evoked.

Produces:
- **Stim-locked** — active and passive, UT and oddball
  - `step8_stimlocked-interp-epo.fif` — concatenated interpolated epochs (all trials,
    labelled with `response_type` / `response_correct` / `trial_idx`)
  - `step8_stimlocked_correct-interp-ave.fif` — evoked, correct trials only (primary)
  - `step8_stimlocked_alltrials-interp-ave.fif` — evoked, all trials
  - for passive, both evokeds are identical (no responses)
- **Response-locked bc-200-100** — active only, UT and oddball (load paper)
  - `step8_responselocked_bc-200-100-interp-epo.fif`
  - `step8_responselocked_bc-200-100-interp-ave.fif`

Flow paper response-locked (CSD) is NOT concatenated — runs are kept
separate because run identity encodes flow condition (high vs low flow).

Outputs go to `derivatives_v2/sub-XXX/concatenated/`.

Last edit: 30.10.2026 @anguyen

In [ ]:
import os
from datetime import datetime
from pathlib import Path

from mne import concatenate_epochs, read_epochs, write_evokeds
from mne.channels import make_standard_montage

date    = datetime.today().strftime('%Y_%m_%d')
montage = make_standard_montage('standard_1020')

ACTIVE_PARTICIPANTS = [
    'P01-G1', 'P02-G2', 'P03-G1', 'P04-G3', 'P05-G2', 'P06-G4', 'P07-G5', 'P08-G6',
    'P10-G8', 'P11-G1', 'P12-G2', 'P13-G3', 'P14-G3', 'P15-G4', 'P16-G4', 'P17-G5',
    'P18-G6', 'P21-G7', 'P22-G7', 'P23-G7', 'P24-G8', 'P25-G1', 'P26-G2', 'P27-G8',
    'P28-G1', 'P29-G3', 'P30-G4', 'P31-G5', 'P32-G6', 'P33-G7', 'P53-G8', 'P55-G1',
    'P56-G8', 'P58-G5', 'P59-G8', 'P60-G6', 'P61-G4', 'P62-G3', 'P63-G5', 'P64-G3',
    'P65-G1', 'P66-G5', 'P67-G3', 'P69-G8', 'P70-G8', 'P72-G1', 'P73-G3', 'P74-G1',
    'P75-G1',
]
PASSIVE_PARTICIPANTS = [
    'S01-G1', 'S02-G2', 'S03-G3', 'S04-G4', 'S05-G5', 'S07-G7', 'S08-G8', 'S09-G1',
    'S11-G3', 'S13-G5', 'S14-G6', 'S15-G1', 'S16-G7', 'S17-G8', 'S18-G1', 'S20-G4',
    'S21-G7', 'S22-G8', 'S23-G3', 'S24-G6', 'S25-G5', 'S26-G2', 'S27-G3', 'S29-G6',
    'S30-G1', 'S31-G4', 'S33-G8', 'S34-G1', 'S35-G2', 'S36-G2', 'S37-G3', 'S39-G3',
    'S41-G5', 'S42-G4', 'S43-G6', 'S45-G8', 'S46-G1', 'S47-G2', 'S49-G4', 'S50-G5',
    'S52-G6', 'S54-G8', 'S55-G1', 'S56-G6', 'S58-G2', 'S59-G3', 'S60-G5',
]

ROOTS = {
    'active':  Path(r'L:\EEG_Flow_data\Study_EEG_Flow_Active\derivatives_v2'),
    'passive': Path(r'L:\EEG_Flow_data\Study_EEG_Flow_Passive\derivatives_v2'),
}


def concat_epochs(root, uid, gid, task, epo_suffix):
    """Load run-1 and run-2 epochs, equalize bads, concatenate, interpolate."""
    pid   = f'sub-{uid}-{gid}'
    paths = [
        root / pid / f'{pid}_task-{task}_run-{run}'
              / f'{pid}_task-{task}_run-{run}_{epo_suffix}'
        for run in [1, 2]
    ]
    epochs_list = []
    for p in paths:
        if p.exists():
            epochs_list.append(read_epochs(p, verbose=False))
            print(f'  Loaded: {p.name} ({len(epochs_list[-1].ch_names)} ch)')
        else:
            print(f'  Missing: {p.name}')

    if not epochs_list:
        raise FileNotFoundError(
            f'No epoch files found for {pid} task-{task}'
        )

    # if channel counts differ, keep only shared channels
    if len(epochs_list) > 1:
        ch_sets = [set(ep.ch_names) for ep in epochs_list]
        common_chs = sorted(ch_sets[0].intersection(*ch_sets[1:]))
        if len(common_chs) < max(len(ep.ch_names) for ep in epochs_list):
            print(f'  Channel mismatch — keeping {len(common_chs)} common channels')
            epochs_list = [ep.pick(common_chs) for ep in epochs_list]

    # equalize bad channels across runs
    all_bads = set()
    for ep in epochs_list:
        all_bads.update(ep.info['bads'])
    for ep in epochs_list:
        ep.info['bads'] = list(all_bads)

    # handle channel count mismatches (e.g. missing game stream in one run)
    if len(epochs_list) > 1:
        ch_sets    = [set(ep.ch_names) for ep in epochs_list]
        common_chs = sorted(set.intersection(*ch_sets))
        if len(common_chs) < max(len(ep.ch_names) for ep in epochs_list):
            dropped = sorted(set.union(*ch_sets) - set(common_chs))
            print(f'  WARNING: channel mismatch — dropping {len(dropped)} channels: {dropped}')
            print(f'  Keeping {len(common_chs)} common channels')
            epochs_list = [ep.pick(common_chs) for ep in epochs_list]

    epochs = (epochs_list[0] if len(epochs_list) == 1
              else concatenate_epochs(epochs_list))

    if epochs.info['bads']:
        epochs.set_montage(montage, on_missing='ignore')
        epochs.interpolate_bads(reset_bads=True, verbose=False)

    return epochs

## 1. Stim-locked — active and passive, UT and oddball

In [ ]:
TRIAL_SETS = ['correct', 'alltrials']

for cohort, participants in [
    ('active',  ACTIVE_PARTICIPANTS),
    ('passive', PASSIVE_PARTICIPANTS),
]:
    root = ROOTS[cohort]

    for pid_group in participants:
        uid, gid = pid_group.split('-')
        pid      = f'sub-{uid}-{gid}'

        for task in ['UT', 'oddball']:
            print(f"\n{'='*70}")
            print(f'{cohort.upper()} | {pid} | task-{task} | stim-locked')

            out_dir = root / pid / 'concatenated'
            out_dir.mkdir(exist_ok=True)
            epo_out  = out_dir / f'{pid}_task-{task}_step8_stimlocked-interp-epo.fif'
            ave_outs = {
                trials: out_dir / f'{pid}_task-{task}_step8_stimlocked_{trials}-interp-ave.fif'
                for trials in TRIAL_SETS
            }

            if epo_out.exists() and all(f.exists() for f in ave_outs.values()):
                print('  Already exists — skipping.')
                continue

            try:
                epochs = concat_epochs(
                    root, uid, gid, task,
                    epo_suffix='step8_stimlocked-epo.fif'
                )
            except FileNotFoundError as e:
                print(f'  SKIP: {e}')
                continue

            # metadata must have survived concatenation
            assert epochs.metadata is not None and 'response_correct' in epochs.metadata

            if not epo_out.exists():
                epochs.save(epo_out, overwrite=False)
                print(f'  Saved: {epo_out.name}')

            is_correct = epochs.metadata['response_correct'].to_numpy(bool)
            selections = {'correct': epochs[is_correct], 'alltrials': epochs}
            for trials, ave_out in ave_outs.items():
                if ave_out.exists():
                    continue
                epo = selections[trials]
                evokeds = [
                    epo[cond].average(picks='all')
                    for cond in epo.event_id
                    if len(epo[cond]) > 0
                ]
                write_evokeds(ave_out, evokeds)
                print(f'  Saved: {ave_out.name} '
                      f'({", ".join(f"{e.comment}={e.nave}" for e in evokeds)})')

## 2. Response-locked bc-200-100 — active only, UT and oddball (load paper)

Flow paper response-locked (CSD) is NOT concatenated — run identity
encodes the flow condition (high vs low flow depending on group mapping).

In [ ]:
root = ROOTS['active']

for pid_group in ACTIVE_PARTICIPANTS:
    uid, gid = pid_group.split('-')
    pid      = f'sub-{uid}-{gid}'

    for task in ['UT', 'oddball']:
        print(f"\n{'='*70}")
        print(f'ACTIVE | {pid} | task-{task} | response-locked bc-200-100')

        out_dir = root / pid / 'concatenated'
        out_dir.mkdir(exist_ok=True)
        epo_out = out_dir / f'{pid}_task-{task}_step8_responselocked_bc-200-100-interp-epo.fif'
        ave_out = out_dir / f'{pid}_task-{task}_step8_responselocked_bc-200-100-interp-ave.fif'

        if epo_out.exists() and ave_out.exists():
            print('  Already exists — skipping.')
            continue

        try:
            epochs = concat_epochs(
                root, uid, gid, task,
                epo_suffix='step8_responselocked_bc-200-100-epo.fif'
            )
        except FileNotFoundError as e:
            print(f'  SKIP: {e}')
            continue

        if not epo_out.exists():
            epochs.save(epo_out, overwrite=False)
            print(f'  Saved: {epo_out.name}')

        evoked = epochs['response'].average(picks='all')
        if not ave_out.exists():
            write_evokeds(ave_out, [evoked])
            print(f'  Saved: {ave_out.name}')